# Movie recommendations

Complete ML pipeline of a movie recommendation system using collaborative filtering.

## Dataset

Creating a synthetic movie ratings

In [1]:
import time
import numpy as np
import pandas as pd
from user_based import UserBasedRecommender
from item_based import ItemBasedRecommender
from matrix_factor import MatrixFactorization

np.random.seed(42)

# Generate synthetic data
n_users = 1000
n_movies = 500
n_ratings = 10000

user_ids = np.random.randint(0, n_users, n_ratings)
movie_ids = np.random.randint(0, n_movies, n_ratings)
ratings = np.random.randint(1, 6, n_ratings)  # 1-5 stars

# Create DataFrame
df = pd.DataFrame({
    'user_id': user_ids,
    'movie_id': movie_ids,
    'rating': ratings
})

# Remove duplicates (user can't rate same movie twice)
df = df.drop_duplicates(subset=['user_id', 'movie_id'])

print(f"Total ratings: {len(df)}")
df.to_csv('movie_ratings.csv', index=False)

Total ratings: 9910


## Evaluation & Analysis

### Tools

In [2]:
def train_test_split_temporal(df, test_size=0.2):
    """
    Split data chronologically (simulate real-world scenario).
    
    Assume ratings are ordered by time.
    """
    split_idx = int(len(df) * (1 - test_size))
    
    train_df = df.iloc[:split_idx]
    test_df = df.iloc[split_idx:]
    
    return train_df, test_df


def compute_rmse(y_true, y_pred):
    """Root Mean Squared Error"""
    total_error = sum((y_true - y_pred)**2)
    return np.sqrt(total_error / len(y_true))


def compute_precision_at_k(recommendations, actual_liked_movies, k=5):
    """
    Of top k recommendations, what fraction did user actually like?
    """
    if len(recommendations) == 0:
        return 0.0
    
    # Get top k movie ids
    top_k = [movie_id for movie_id, _ in recommendations[:k]]
    
    # Count how many are in actual liked movies
    hits = sum(1 for movie_id in top_k if movie_id in actual_liked_movies)

    return hits / k

### Evaluation

In [3]:
def evaluate_all_methods(train_df, test_df):
    """
    Train all three recommenders and compare.
    
    Report:
    - RMSE
    - Precision@5
    - Training time
    - Inference time
    """
    print("=" * 60)
    print("EVALUATING ALL RECOMMENDATION METHODS")
    print("=" * 60)
    
    # Prepare test set
    # Get actual ratings from test set for RMSE calculations
    test_ratings = {} # {(user_id, movie_id): rating}
    for u, m, r in test_df[['user_id', 'movie_id', 'rating']].values:
        test_ratings[(int(u), int(m))] = int(r)
        
    # Get users who have high ratings in test set (for precision)
    test_liked = {} # {user_id: set of liked movies}
    for u, m, r in test_df[['user_id', 'movie_id', 'rating']].values:
        if r >= 4: # Consider 4-5 stars as "liked"
            if u not in test_liked:
                test_liked[u] = set()
            
            test_liked[u].add(int(m))
            
    sample_users = list(test_liked.keys())[:100] # Evaluate on 100 users
    
    results = {}
    
    
    # =====================================================
    # 1. USER-BASED COLLABORATIVE FILTERING
    # =====================================================
    print("\n1. User-Based Collaborative Filtering")
    print("-" * 60)
    
    start = time.time()
    user_rec = UserBasedRecommender(train_df)
    train_time_user = time.time() - start
    print(f"Training time: {train_time_user:.2f}s")
    
    # Evaluate RMSE
    predictions = []
    actuals = []
    for (u, m), r in test_ratings.items():
        try:
            # Predict this rating
            similar_users = user_rec.find_similar_users(u, k=10)
            if len(similar_users) > 0:
                # Weighted average of similar users' ratings for this movie
                weighted_sum = 0
                sim_sum = 0
                for sim_user, sim_score in similar_users:
                    if m in user_rec.ratings.get(sim_user, {}):
                        weighted_sum += sim_score * user_rec.ratings[sim_user][m]
                        sim_sum += sim_score
                        
                if sim_sum > 0:
                    pred = weighted_sum / sim_sum
                    predictions.append(pred)
                    actuals.append(r)
        except:
            pass
    
    rmse_user = compute_rmse(np.array(actuals), np.array(predictions)) if len(actuals) > 0 else float('inf')
    print(f"RMSE: {rmse_user:.4f} (on {len(actuals)} test ratings)")
    
    # Evaluate Precision@5
    start = time.time()
    precisions = []
    for user_id in sample_users:
        recs = user_rec.recommend(user_id, n=5)
        if user_id in test_liked:
            prec = compute_precision_at_k(recs, test_liked[user_id], k=5)
            precisions.append(prec)
    
    infer_time_user = (time.time() - start) / len(sample_users)
    precision_user = np.mean(precisions) if len(precisions) > 0 else 0
    print(f"Precision@5: {precision_user:.4f}")
    print(f"Avg inference time: {infer_time_user * 1000:.2f}ms per user")
    
    results['User-Based'] = {
        'rmse': rmse_user,
        'precision': precision_user,
        'train_time': train_time_user,
        'infer_time': infer_time_user
    }
    
    # =====================================================
    # 2. ITEM-BASED COLLABORATIVE FILTERING
    # =====================================================
    print("\n2. Item-Based Collaborative Filtering")
    print("-" * 60)
    
    start = time.time()
    item_rec = ItemBasedRecommender(train_df)
    train_time_item = time.time() - start
    print(f"Training time: {train_time_item:.2f}s")
        
    # Evaluate Precision@5 
    start = time.time()
    precisions = []
    for user_id in sample_users:
        recs = item_rec.recommend(user_id, n=5)
        if user_id in test_liked:
            prec = compute_precision_at_k(recs, test_liked[user_id], k=5)
            precisions.append(prec)
            
    infer_time_item = (time.time() - start) / len(sample_users)
    precision_item = np.mean(precisions) if len(precisions) > 0 else 0
    print(f"Precision@5: {precision_item:.4f}")
    print(f"Avg inference time: {infer_time_item * 1000:.2f}ms per user")
    
    results['Item-Based'] = {
        'rmse': None,
        'precision': precision_item,
        'train_time': train_time_item,
        'infer_time': infer_time_item
    }

    # =====================================================
    # 3. MATRIX FACTORIZATION
    # =====================================================
    print("\n3. Matrix Factorization")
    print("-" * 60)
    
    n_users = train_df['user_id'].max() + 1
    n_movies = train_df['movie_id'].max() + 1
    
    start = time.time()
    mf = MatrixFactorization(n_users=n_users, n_movies=n_movies, k=20, learning_rate=0.01, reg=0.01)
    mf.train(train_df, epochs=50)
    train_time_mf = time.time() - start
    print(f"Training time: {train_time_mf:.2f}s")
    
    # Evaluate RMSE
    predictions = []
    actuals = []
    for (u, m), r in test_ratings.items():
        try:
            pred = mf.predict(u, m)
            predictions.append(pred)
            actuals.append(r)
        except:
            pass
        
    rmse_mf = compute_rmse(np.array(actuals), np.array(predictions)) if len(actuals) > 0 else float('inf')
    print(f"RMSE: {rmse_mf:.4f} (on {len(actuals)} test ratings)")
    
    # Evaluate Precision@5
    start = time.time()
    precisions = []
    for user_id in sample_users:
        # Get seen movies
        seen_movies = set(train_df[train_df['user_id'] == user_id]['movie_id'].values)
        recs = mf.recommend(user_id, seen_movies, n=5)
        if user_id in test_liked:
            prec = compute_precision_at_k(recs, test_liked[user_id], k=5)
            precisions.append(prec)
    
    infer_time_mf = (time.time() - start) / len(sample_users)
    precision_mf = np.mean(precisions) if len(precisions) > 0 else 0
    print(f"Precision@5: {precision_mf:.4f}")
    print(f"Avg inference time: {infer_time_mf * 1000:.2f}ms per user")
    
    results['Matrix Factorization'] = {
        'rmse': rmse_mf,
        'precision': precision_mf,
        'train_time': train_time_mf,
        'infer_time': infer_time_mf
    }
    
    # =====================================================
    # SUMMARY
    # =====================================================
    print("\n" + "=" * 60)
    print("SUMMARY: COMPARISON OF ALL METHODS")
    print("=" * 60)
    
    print(f"\n{'Method':<25} {'RMSE':<10} {'Prec@5':<10} {'Train(s)':<12} {'Infer(ms)':<12}")
    print("-" * 70)
    
    for method, metrics in results.items():
        rmse_str = f"{metrics['rmse']:.4f}" if metrics['rmse'] is not None else "N/A"
        print(f"{method:<25} {rmse_str:<10} {metrics['precision']:<10.4f} "
              f"{metrics['train_time']:<12.2f} {metrics['infer_time'] * 1000:<12.2f}")
        
    print("\n" + "=" * 60)

    return results

### Usage

In [4]:
# Split
train_df, test_df = train_test_split_temporal(df, test_size=0.2)

print(f"Train size: {len(train_df)}")
print(f"Test size: {len(test_df)}\n")

# Evaluate all methods
results = evaluate_all_methods(train_df, test_df)

Train size: 7928
Test size: 1982

EVALUATING ALL RECOMMENDATION METHODS

1. User-Based Collaborative Filtering
------------------------------------------------------------
Training time: 0.01s
RMSE: 1.9054 (on 285 test ratings)
Precision@5: 0.0000
Avg inference time: 1.92ms per user

2. Item-Based Collaborative Filtering
------------------------------------------------------------
Training time: 0.01s
Precision@5: 0.0060
Avg inference time: 4.16ms per user

3. Matrix Factorization
------------------------------------------------------------
Epoch 10/50, Loss: 0.9459
Epoch 20/50, Loss: 0.6036
Epoch 30/50, Loss: 0.3717
Epoch 40/50, Loss: 0.2400
Epoch 50/50, Loss: 0.1638
Training time: 3.33s
RMSE: 1.7577 (on 1982 test ratings)
Precision@5: 0.0060
Avg inference time: 0.31ms per user

SUMMARY: COMPARISON OF ALL METHODS

Method                    RMSE       Prec@5     Train(s)     Infer(ms)   
----------------------------------------------------------------------
User-Based                1.